![Setup and Local Stack](assets/banner-00.svg)

Every notebook in this lab sends real telemetry to a real backend. This one gets that backend running on your machine and proves each piece works before we write a single span.

**You will:**
1. Start a self-hosted Langfuse with Docker Compose
2. Understand what each of its six containers does
3. Confirm your API keys and OpenAI access

**Time:** about 5 minutes, most of it waiting for images to download the first time.

## The stack we are about to start

![Langfuse stack](assets/langfuse-stack.svg)

| Container | Role | Why it exists |
|---|---|---|
| `langfuse-web` | UI and ingestion API on port 3000 | Receives spans from SDKs and OTLP exporters, serves the dashboard |
| `langfuse-worker` | Background processor | Turns raw events into observations, computes token cost, runs evaluations |
| `redis` | Queue and cache | Decouples fast ingestion from slower processing |
| `minio` | S3-compatible blob store | Keeps raw event payloads and media durable before processing |
| `clickhouse` | Columnar analytics database | Stores traces, observations and scores; fast aggregation for dashboards |
| `postgres` | Relational database | Users, projects, API keys, prompts, settings |

This split is the same pattern most observability backends use: a thin, fast ingestion edge, a queue, async workers, and a column store tuned for "group by and percentile" queries.

## 1. Start Langfuse

The compose file lives in `langfuse/`. Its `.env` uses Langfuse's **headless initialization**, which creates a user, an organization, a project and a pair of API keys on first boot. No clicking through a signup form.

```text
login     admin@observability.lab
password  observability123
keys      pk-lf-local-demo-public / sk-lf-local-demo-secret
```

These are local demo credentials. Change them before you run this anywhere other than your laptop.

In [1]:
import subprocess
from pathlib import Path

ROOT = Path.cwd().parent
LANGFUSE_DIR = ROOT / "langfuse"

if not (LANGFUSE_DIR / ".env").exists():
    (LANGFUSE_DIR / ".env").write_text((LANGFUSE_DIR / ".env.example").read_text())

result = subprocess.run(
    ["docker", "compose", "up", "-d"], cwd=LANGFUSE_DIR, capture_output=True, text=True
)
print(result.stderr.strip().splitlines()[-1] if result.stderr else result.stdout)

 Container langfuse-minio-1  Healthy


## 2. Wait until it is healthy

The first start runs database migrations, which takes one to three minutes. The health endpoint answers once the web container is ready.

In [2]:
import time

import requests

from lab import callout

for attempt in range(60):
    try:
        health = requests.get("http://localhost:3000/api/public/health", timeout=3).json()
        break
    except requests.RequestException:
        time.sleep(5)
else:
    raise RuntimeError("Langfuse did not become healthy. Run `docker compose logs langfuse-web` in langfuse/.")

callout("Langfuse is up", f"status <b>{health['status']}</b>, version <b>{health['version']}</b> at http://localhost:3000")

In [3]:
import json

import pandas as pd

ps = subprocess.run(
    ["docker", "compose", "ps", "--format", "json"], cwd=LANGFUSE_DIR, capture_output=True, text=True
).stdout
rows = [json.loads(line) for line in ps.strip().splitlines()]
pd.DataFrame(rows)[["Service", "State", "Status", "Publishers"]].assign(
    Publishers=lambda d: d["Publishers"].map(
        lambda ps: ", ".join(f"{p['URL']}:{p['PublishedPort']}" for p in (ps or []) if p.get("PublishedPort"))
    )
).rename(columns={"Publishers": "Ports"})

,Service,State,Status,Ports
0,clickhouse,running,Up 2 hours (healthy),"127.0.0.1:8123, 127.0.0.1:9000"
1,langfuse-web,running,Up 2 hours,0.0.0.0:3000
2,langfuse-worker,running,Up 2 hours,127.0.0.1:3030
3,minio,running,Up 2 hours (healthy),"0.0.0.0:9090, 127.0.0.1:9091"
4,postgres,running,Up 2 hours (healthy),127.0.0.1:5433
5,redis,running,Up 2 hours (healthy),127.0.0.1:6379


## 3. Check credentials

All notebooks read the repo root `.env`. We print only whether each variable is set, never its value.

In [4]:
import os

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

checks = {
    "OPENAI_API_KEY": "every notebook",
    "LANGFUSE_PUBLIC_KEY": "notebooks 02 and 04",
    "LANGFUSE_SECRET_KEY": "notebooks 02 and 04",
    "LANGFUSE_BASE_URL": "notebooks 02 and 04",
    "LANGSMITH_API_KEY": "notebook 03 (optional)",
}
pd.DataFrame(
    [{"variable": k, "set": "yes" if os.getenv(k) else "no", "used by": v} for k, v in checks.items()]
)

,variable,set,used by
0,OPENAI_API_KEY,yes,every notebook
1,LANGFUSE_PUBLIC_KEY,yes,notebooks 02 and 04
2,LANGFUSE_SECRET_KEY,yes,notebooks 02 and 04
3,LANGFUSE_BASE_URL,yes,notebooks 02 and 04
4,LANGSMITH_API_KEY,yes,notebook 03 (optional)


Langfuse authenticates API calls with HTTP Basic auth: public key as the username, secret key as the password. A quick authenticated call proves the keys match the project the stack created.

In [5]:
r = requests.get(
    f"{os.environ['LANGFUSE_BASE_URL']}/api/public/projects",
    auth=(os.environ["LANGFUSE_PUBLIC_KEY"], os.environ["LANGFUSE_SECRET_KEY"]),
    timeout=10,
)
r.raise_for_status()
project = r.json()["data"][0]
callout("Keys accepted", f"project <b>{project['name']}</b> (id <code>{project['id']}</code>)")

## 4. One OpenAI call

The agent in every notebook uses `gpt-4o-mini`. One tiny request confirms the key and network path.

In [6]:
from openai import OpenAI

client = OpenAI()
reply = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Reply with exactly: ready"}],
    max_tokens=5,
)
callout(
    "OpenAI reachable",
    f"model said <b>{reply.choices[0].message.content}</b> using {reply.usage.total_tokens} tokens",
)

## You are ready

Open **http://localhost:3000** and sign in with the demo login. The project is empty for now. By the end of notebook 02 it will have your first traces.

| Next | What it covers |
|---|---|
| [01 What is observability?](01_what_is_observability.ipynb) | Logs, metrics, traces and spans in plain words |
| [02 Langfuse](02_langfuse.ipynb) | Your first trace in a real dashboard |
| [03 LangSmith](03_langsmith.ipynb) | The same function in another tool |
| [04 Other tools](04_other_tools.ipynb) | OpenTelemetry, Datadog, Dynatrace and how to choose |

To stop everything later: `docker compose down` in `langfuse/`. Add `-v` to wipe the data too.